### Libraries and Depencies

In [ ]:
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from transformers import logging as hf_logging
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import logging
import warnings
import time
import re
import os
import spacy

from pinecone import Pinecone, ServerlessSpec
from rank_bm25 import BM25Okapi



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 32.7 MB/s eta 0:00:00


In [ ]:
HF_TOKEN = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN') 
print(f"Hugging Face API Token: {'Set' if HF_TOKEN else 'Not Set'}")

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

hf_logging.set_verbosity_error()
try:
    hf_logging.disable_progress_bar()
except Exception:
    pass

logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)

warnings.filterwarnings('ignore', message='Both `max_new_tokens`')
warnings.filterwarnings('ignore', message='Setting `pad_token_id`')

try:
    nlp = spacy.load('en_core_web_sm')
except Exception:
    nlp = None

_GLOBAL_MODELS = globals().setdefault('_GLOBAL_MODELS', {})
_RETRIEVAL_CACHE = globals().setdefault('_RETRIEVAL_CACHE', {})
_RUNTIME_FLAGS = globals().setdefault(
    '_RUNTIME_FLAGS',
    {'pinecone_warning_printed': False, 'disable_hf_judge_remote': False}
)

Hugging Face API Token: Set


In [5]:
CORPUS_PATH = Path.cwd() / 'support_1000.parquet'

CHUNKING_METHOD = 'recursive'   # 'arbitary' | 'recursive' | 'semantic'
TOP_K = 5

# Bump this when chunking logic changes so old caches are not reused accidentally.
CHUNK_CACHE_VERSION = 'v6'
SEMANTIC_MAX_CHARS = 700
SEMANTIC_MIN_CHARS = 220
SEMANTIC_OVERLAP_SENTENCES = 1

CHUNK_CACHE_DIR = Path.cwd() / 'chunk_cache'
CHUNK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

### Chunking Strategy

In [6]:
# Arbitary Size Chunking
def Arbitary_chunking(text, chunk_size=512, overlap=64):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

In [7]:
# Recursive Chunking
def Recursive_chunking(text, target_size=800):
    sections = text.split('\n## ')
    chunks = []
    for section in sections:
        if len(section) > 1000:
            paragraphs = section.split('\n\n')
            current_chunk = ""
            for para in paragraphs:
                if len(current_chunk + para) < target_size:
                    current_chunk += para + "\n\n"
                else:
                    if current_chunk.strip():
                        chunks.append(current_chunk.strip())
                    current_chunk = para + "\n\n"
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
        else:
            if section.strip():
                chunks.append(section.strip())
    return chunks

In [8]:
def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '')).strip()

def _split_sentences(text: str) -> List[str]:
    if not text:
        return []
    try:
        if nlp is not None:
            doc = nlp(text)
            return [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    except Exception:
        pass
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

# Semantic Chunking
def Semantic_chunking(text, max_chars=None, min_chars=None, overlap_sentences=None):
    max_chars = max_chars or SEMANTIC_MAX_CHARS
    min_chars = min_chars or SEMANTIC_MIN_CHARS
    overlap_sentences = SEMANTIC_OVERLAP_SENTENCES if overlap_sentences is None else overlap_sentences

    text = normalize_text(text)
    sentences = _split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current_sentences = []
    current_len = 0

    for sent in sentences:
        sent_len = len(sent) + 1
        if current_sentences and current_len + sent_len > max_chars:
            candidate = ' '.join(current_sentences).strip()
            if candidate:
                if chunks and len(candidate) < min_chars:
                    chunks[-1] = f"{chunks[-1]} {candidate}".strip()
                else:
                    chunks.append(candidate)

            if overlap_sentences > 0 and len(current_sentences) > 0:
                current_sentences = current_sentences[-overlap_sentences:]
                current_len = sum(len(x) + 1 for x in current_sentences)
            else:
                current_sentences = []
                current_len = 0

        current_sentences.append(sent)
        current_len += sent_len

    if current_sentences:
        candidate = ' '.join(current_sentences).strip()
        if candidate:
            if chunks and len(candidate) < min_chars:
                chunks[-1] = f"{chunks[-1]} {candidate}".strip()
            else:
                chunks.append(candidate)

    if len(chunks) >= 2 and len(chunks[-1]) < min_chars:
        chunks[-2] = f"{chunks[-2]} {chunks[-1]}".strip()
        chunks.pop()

    return chunks

def chunk_document(text, method='recursive'):
    clean_text = normalize_text(text)
    if method == 'arbitary':
        return Arbitary_chunking(clean_text)
    if method == 'semantic':
        return Semantic_chunking(clean_text)

    # Recursive first, then semantically split oversized parts for better retrieval focus.
    recursive_chunks = Recursive_chunking(clean_text, target_size=SEMANTIC_MAX_CHARS)
    refined = []
    for chunk in recursive_chunks:
        if len(chunk) > int(SEMANTIC_MAX_CHARS * 1.25):
            refined.extend(Semantic_chunking(chunk))
        else:
            refined.append(chunk.strip())
    return [c for c in refined if c and c.strip()]

# Corpus Loading and Chunking

In [9]:
def _corpus_signature(corpus_path: Path) -> str:
    import hashlib

    h = hashlib.sha1()
    corpus_path = Path(corpus_path)

    if corpus_path.is_file():
        st = corpus_path.stat()
        h.update(f"{corpus_path.resolve()}|{st.st_mtime_ns}|{st.st_size}".encode('utf-8'))
        return h.hexdigest()[:12]

    if corpus_path.is_dir():
        supported = {'.txt', '.md', '.csv', '.parquet'}
        for fp in sorted(corpus_path.rglob('*')):
            if fp.is_file() and fp.suffix.lower() in supported:
                st = fp.stat()
                h.update(f"{fp.resolve()}|{st.st_mtime_ns}|{st.st_size}".encode('utf-8'))
        return h.hexdigest()[:12]

    h.update(str(corpus_path).encode('utf-8'))
    return h.hexdigest()[:12]

def get_chunk_cache_path(corpus_path, chunking_method='recursive'):
    import hashlib

    corpus_path = Path(corpus_path)
    corpus_name = corpus_path.stem if corpus_path.is_file() else corpus_path.name
    corpus_name = re.sub(r'[^a-zA-Z0-9._-]+', '_', corpus_name or 'corpus').strip('_').lower()
    resolved = str(corpus_path.resolve()) if corpus_path.exists() else str(corpus_path)
    source_sig = _corpus_signature(corpus_path)
    chunk_cfg = f"{CHUNK_CACHE_VERSION}|{chunking_method}|{SEMANTIC_MAX_CHARS}|{SEMANTIC_MIN_CHARS}|{SEMANTIC_OVERLAP_SENTENCES}"
    digest = hashlib.sha1(f"{resolved}|{source_sig}|{chunk_cfg}".encode('utf-8')).hexdigest()[:12]
    return CHUNK_CACHE_DIR / f"{corpus_name}_{chunking_method}_{CHUNK_CACHE_VERSION}_{digest}.jsonl"

def read_corpus_documents(corpus_path):
    corpus_path = Path(corpus_path)
    docs = []

    def _normalize_cell(v):
        if pd.isna(v):
            return ''
        return normalize_text(str(v))

    def _info_score(text: str) -> float:
        t = normalize_text(text).lower()
        if not t:
            return 0.0
        tokens = [w for w in re.split(r'\W+', t) if w]
        unique = len(set(tokens))
        keyword_hits = sum(1 for kw in ['step', 'settings', 'ios', 'android', 'server', 'vpn', 'configure'] if kw in t)
        return unique + 8.0 * keyword_hits

    def _load_text_file(fp):
        text = fp.read_text(encoding='utf-8', errors='ignore').strip()
        if text:
            docs.append({'source': str(fp), 'text': text})

    def _load_tabular_rows(df, fp, ignore_first_column=False):
        all_cols = list(df.columns)
        if ignore_first_column and all_cols:
            df = df.iloc[:, 1:].copy()
            all_cols = list(df.columns)

        if not all_cols:
            return

        def _pick_cols(preferred_names):
            out = []
            for name in preferred_names:
                match = next((c for c in all_cols if str(c).lower() == name), None)
                if match and match not in out:
                    out.append(match)
            return out

        main_cols = _pick_cols([
            'ki_text', 'text', 'content', 'body', 'description', 'details',
            'instructions', 'procedure', 'answer', 'passage', 'alt_ki_text'
        ])
        meta_cols = _pick_cols(['ki_topic', 'topic', 'title', 'category', 'subtopic'])

        object_cols = [
            c for c in df.columns
            if pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c])
        ]
        if not main_cols:
            ignore_tokens = ('id', 'uuid', 'timestamp', 'created', 'updated')
            filtered = [c for c in object_cols if not any(tok in str(c).lower() for tok in ignore_tokens)]
            main_cols = filtered if filtered else (object_cols if object_cols else list(df.columns))

        for idx, row in df.iterrows():
            topic_parts = [_normalize_cell(row.get(c, None)) for c in meta_cols]
            topic_parts = [x for x in topic_parts if x]

            candidates = [_normalize_cell(row.get(c, None)) for c in main_cols]
            candidates = [x for x in candidates if x]
            if not candidates:
                continue

            best_content = max(candidates, key=_info_score)
            content_text = best_content

            topic_text = ' | '.join(topic_parts)
            row_text = f"Topic: {topic_text}\n{content_text}" if topic_text else content_text

            docs.append({
                'source': f'{fp}#row={idx}',
                'text': row_text
            })

    def _load_csv_file(fp):
        df = pd.read_csv(fp)
        _load_tabular_rows(df, fp, ignore_first_column=False)

    def _load_parquet_file(fp):
        try:
            df = pd.read_parquet(fp)
        except ImportError as exc:
            raise ImportError("Reading parquet files requires 'pyarrow' or 'fastparquet'.") from exc
        _load_tabular_rows(df, fp, ignore_first_column=True)

    if corpus_path.is_file():
        suffix = corpus_path.suffix.lower()
        if suffix in ['.txt', '.md']:
            _load_text_file(corpus_path)
        elif suffix == '.csv':
            _load_csv_file(corpus_path)
        elif suffix == '.parquet':
            _load_parquet_file(corpus_path)
        return docs

    for pattern in ['*.txt', '*.md']:
        for fp in corpus_path.rglob(pattern):
            _load_text_file(fp)

    for fp in corpus_path.rglob('*.csv'):
        _load_csv_file(fp)

    for fp in corpus_path.rglob('*.parquet'):
        _load_parquet_file(fp)

    return docs

In [10]:
def build_chunks_from_docs(docs, method='recursive'):
    chunks = []
    chunk_counter = 0
    for doc in docs:
        local_chunks = chunk_document(doc['text'], method=method)
        for ch in local_chunks:
            chunks.append({
                'id': f'ch_{chunk_counter}',
                'text': ch,
                'source': doc['source']
            })
            chunk_counter += 1
    return chunks

### Document Retrival

In [11]:
def get_cached_embedding_model(model_name: str):
    cache_key = f'embedding::{model_name}'
    model = _GLOBAL_MODELS.get(cache_key)
    if model is None:
        model = SentenceTransformer(model_name)
        _GLOBAL_MODELS[cache_key] = model
    return model

def get_cached_classifier(model_name: str = 'cross-encoder/nli-deberta-v3-small'):
    cache_key = f'classifier::{model_name}'
    clf = _GLOBAL_MODELS.get(cache_key)
    if clf is None:
        clf = pipeline('zero-shot-classification', model=model_name)
        _GLOBAL_MODELS[cache_key] = clf
    return clf

class HybridRetriever:
    """
    Hybrid retrieval with BM25 + Semantic (Pinecone/local) using RRF + optional re-ranking.
    """

    def __init__(self, index_name='rag-assignment3-index', embedding_model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
        self.embedding_model = get_cached_embedding_model(embedding_model_name)
        self.index_name = index_name
        self.pc_client = None
        self.pc_index = None

        self.bm25_model = None
        self.bm25_chunks = []
        self.semantic_matrix = None

        self.intent_labels = ['factual query', 'procedural steps', 'conceptual explanation']
        self.classifier = get_cached_classifier('cross-encoder/nli-deberta-v3-small')

        pinecone_api_key = os.getenv('PINECONE_API_KEY')
        pinecone_env = os.getenv('PINECONE_ENVIRONMENT', 'us-east-1')

        if Pinecone is not None and pinecone_api_key:
            self.pc_client = Pinecone(api_key=pinecone_api_key)
            existing_indexes = [idx.name for idx in self.pc_client.list_indexes()]
            if index_name not in existing_indexes:
                self.pc_client.create_index(
                    name=index_name,
                    dimension=384,
                    metric='cosine',
                    spec=ServerlessSpec(cloud='aws', region=pinecone_env)
                )
            self.pc_index = self.pc_client.Index(index_name)

    def classify_query(self, query: str) -> str:
        prediction = self.classifier(query, self.intent_labels)
        top_label = prediction['labels'][0].lower()
        if 'factual' in top_label:
            return 'factual'
        if 'procedural' in top_label:
            return 'procedural'
        return 'conceptual'

    def set_bm25_corpus(self, corpus_chunks: list):
        """Build BM25 corpus and precompute local semantic vectors."""
        self.corpus = corpus_chunks
        if BM25Okapi is None:
            self.bm25 = None
        else:
            tokenized_corpus = [
                [word for word in re.split(r'\W+', chunk.get('text', '').lower()) if word]
                for chunk in self.corpus
            ]
            self.bm25 = BM25Okapi(tokenized_corpus) if tokenized_corpus else None

        texts = [chunk.get('text', '') for chunk in self.corpus]
        if texts:
            self.semantic_matrix = np.array(self.embedding_model.encode(texts, show_progress_bar=False))
        else:
            self.semantic_matrix = None

    def _bm25_search(self, query: str, top_k: int) -> list:
        """BM25 retrieval using the same regex tokenization strategy as indexing."""
        if not getattr(self, 'bm25', None):
            return []

        tokenized_query = [word for word in re.split(r'\W+', query.lower()) if word]
        scores = self.bm25.get_scores(tokenized_query)

        top_indices = np.argsort(scores)[::-1][:top_k]
        return [
            {
                'id': self.corpus[i]['id'],
                'text': self.corpus[i]['text'],
                'source': self.corpus[i].get('source', 'unknown'),
                'score': float(scores[i]),
                'search_type': 'keyword'
            }
            for i in top_indices
        ]

    def _semantic_search(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        if self.pc_index is not None:
            try:
                query_vector = self.embedding_model.encode(query).tolist()
                response = self.pc_index.query(
                    vector=query_vector,
                    top_k=top_k,
                    include_metadata=True
                )

                results = []
                for m in response.matches:
                    meta = m.metadata or {}
                    results.append({
                        'id': m.id,
                        'text': meta.get('text', ''),
                        'source': meta.get('source', 'unknown'),
                        'score': float(m.score),
                        'search_type': 'semantic'
                    })
                return results
            except Exception:
                pass

        if self.semantic_matrix is None or len(getattr(self, 'corpus', [])) == 0:
            return []

        query_vector = self.embedding_model.encode(query)
        similarities = cosine_similarity([query_vector], self.semantic_matrix)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [
            {
                'id': self.corpus[i]['id'],
                'text': self.corpus[i]['text'],
                'source': self.corpus[i].get('source', 'unknown'),
                'score': float(similarities[i]),
                'search_type': 'semantic'
            }
            for i in top_indices
        ]

    def _rrf_fusion(self, keyword_results, semantic_results, k=60):
        rrf_scores = {}
        merged_docs = {}

        for rank, doc in enumerate(keyword_results, start=1):
            doc_id = doc['id']
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            merged_docs[doc_id] = doc

        for rank, doc in enumerate(semantic_results, start=1):
            doc_id = doc['id']
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            if doc_id in merged_docs:
                merged_docs[doc_id]['search_type'] = 'hybrid'
            else:
                merged_docs[doc_id] = doc

        fused = []
        for doc_id, score in rrf_scores.items():
            doc = dict(merged_docs[doc_id])
            doc['rrf_score'] = score
            fused.append(doc)

        fused.sort(key=lambda x: x.get('rrf_score', 0.0), reverse=True)
        return fused

    def _rerank(self, query: str, results: list, top_k: int) -> list:
        """Optimized to use BATCH vectorization for massive speedup."""
        if not results:
            return []

        query_emb = self.embedding_model.encode(query)
        texts_to_encode = [doc.get('text', '') for doc in results]
        doc_embeddings = self.embedding_model.encode(texts_to_encode)
        similarities = cosine_similarity([query_emb], doc_embeddings)[0]

        for idx, doc in enumerate(results):
            doc['rerank_score'] = similarities[idx]

        results.sort(key=lambda x: x['rerank_score'], reverse=True)
        return results[:top_k]

    def retrieve_hybrid(self, query: str, top_k: int = 5, rerank: bool = True) -> List[Dict[str, Any]]:
        query_type = self.classify_query(query)
        if query_type == 'factual':
            keyword_results = self._bm25_search(query, top_k=8)
            semantic_results = self._semantic_search(query, top_k=4)
        else:
            keyword_results = self._bm25_search(query, top_k=4)
            semantic_results = self._semantic_search(query, top_k=8)

        fused = self._rrf_fusion(keyword_results, semantic_results)
        if rerank:
            return self._rerank(query, fused, top_k=top_k)
        return fused[:top_k]

    def retrieve_semantic_only(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        return self._semantic_search(query, top_k=top_k)

In [12]:
utility_embedding_model = get_cached_embedding_model('all-MiniLM-L6-v2')

def get_embedding(text):
    return utility_embedding_model.encode(text)

def count_tokens(text):
    # Simple approximation for notebook experimentation
    return len(text.split())

def remove_duplicates(retrieved_chunks: list) -> list:
    """Removes duplicate dictionaries from a list based on their 'id'."""
    seen_ids = set()
    unique_chunks = []

    for chunk in retrieved_chunks:
        if chunk['id'] not in seen_ids:
            seen_ids.add(chunk['id'])
            unique_chunks.append(chunk)

    return unique_chunks

def diversify_by_source(retrieved_chunks: list, max_per_source: int = 4) -> list:
    """Limit over-representation of a single source while still keeping enough detail."""
    by_source = {}
    diversified = []
    for chunk in retrieved_chunks:
        src = chunk.get('source', 'unknown')
        by_source[src] = by_source.get(src, 0) + 1
        if by_source[src] <= max_per_source:
            diversified.append(chunk)
    return diversified

def adds_new_info(new_chunk_emb: np.ndarray, selected_embs: list, threshold: float = 0.92) -> bool:
    """
    Checks if a chunk adds new info by comparing pre-calculated embeddings.
    """
    if not selected_embs:
        return True

    # Check similarity against all already selected chunks
    for selected_emb in selected_embs:
        sim = cosine_similarity([new_chunk_emb], [selected_emb])[0][0]
        if sim > threshold:
            return False # Too similar to an existing chunk

    return True

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [13]:
def intelligent_context_selection(retrieved_chunks: list, embedding_model, max_tokens: int = 1500) -> list:
    """
    Select diverse chunks under a token budget and return chunk dictionaries.
    """
    selected_chunks = []
    selected_embs = []  # Cache to store embeddings we've already calculated
    current_length = 0

    for chunk in retrieved_chunks:
        text = chunk.get('text', '')
        if not text:
            continue

        chunk_length = len(text.split())
        if current_length + chunk_length > max_tokens:
            continue

        # Get embedding for the current chunk just ONCE
        new_emb = embedding_model.encode(text)

        # Pass the pre-calculated embeddings to our helper
        if adds_new_info(new_emb, selected_embs):
            selected_chunks.append(chunk)
            selected_embs.append(new_emb)  # Store the embedding so we don't recalculate it later
            current_length += chunk_length

    return selected_chunks

In [14]:
def create_rag_prompt(query, context_chunks):
    context_text = "\n\n".join([
        f"Source {i+1}: {chunk['text']}"
        for i, chunk in enumerate(context_chunks)
    ])

    is_comparative = bool(re.search(r'\b(difference|compare|versus|vs)\b', query.lower()))
    comparative_instruction = (
        "8. If the question is comparative, provide side-by-side points for each item mentioned."
        if is_comparative
        else "8. Structure the answer as concise bullet points."
    )

    prompt = f"""CONTEXT:
{context_text}

QUESTION: {query}

INSTRUCTIONS:
You are an empathetic, professional mental health counselor speaking directly to the patient.
Read the provided context, but DO NOT quote it directly. DO NOT use [Source] tags or sound like a textbook.

Write a warm, cohesive, and supportive response following this exact structure:
1. Start with a compassionate opening paragraph that directly validates their specific feelings (e.g., "I hear how exhausted you are...").
2. Provide 3 practical, actionable steps or coping strategies derived ONLY from the context.
3. Conclude with a supportive closing sentence.

"""
    return prompt

def _clean_generated_answer(text: str) -> str:
    text = normalize_text(text)
    if not text:
        return text

    sentences = _split_sentences(text)
    if not sentences:
        return text[:1800]

    unique = []
    seen = set()
    for s in sentences:
        norm = normalize_text(s)
        if not norm:
            continue
        key = re.sub(r'^\d+[\)\.:\-\s]+', '', norm.lower())
        if key in seen:
            continue
        seen.add(key)
        unique.append(norm)

    cleaned = ' '.join(unique).strip()
    return cleaned[:1800]

def _extractive_fallback_answer(prompt: str, max_points: int = 6) -> str:
    context_match = re.search(r'CONTEXT:\s*(.*?)\s*QUESTION:', prompt, re.DOTALL)
    query_match = re.search(r'QUESTION:\s*(.*?)\s*INSTRUCTIONS:', prompt, re.DOTALL)

    if not context_match or not query_match:
        return 'Not found in provided context.'

    context_block = context_match.group(1).strip()
    query = normalize_text(query_match.group(1))
    if not context_block or not query:
        return 'Not found in provided context.'

    source_items = []
    pattern = re.compile(r'Source\s+(\d+):\s*(.*?)(?=(?:\n\s*Source\s+\d+:)|\Z)', re.DOTALL)
    for m in pattern.finditer(context_block):
        src = int(m.group(1))
        txt = normalize_text(m.group(2))
        if txt:
            source_items.append((src, txt))

    if not source_items:
        return 'Not found in provided context.'

    candidates = []
    for src, txt in source_items:
        sentences = [s for s in _split_sentences(txt) if len(s.split()) >= 4]
        if not sentences:
            sentences = [txt]
        for s in sentences:
            candidates.append((src, normalize_text(s)))

    if not candidates:
        return 'Not found in provided context.'

    emb = get_cached_embedding_model('all-MiniLM-L6-v2')
    q_vec = emb.encode(query)
    sent_texts = [c[1] for c in candidates]
    sent_vecs = np.array(emb.encode(sent_texts, show_progress_bar=False))
    sims = cosine_similarity([q_vec], sent_vecs)[0]

    stop = {
        'what','which','where','when','why','how','the','and','for','with','from','that','this',
        'are','is','was','were','into','onto','about','between','versus','vs','difference','setup',
        'process','configure','configuring','company','email','device'
    }
    q_terms = [w for w in re.split(r'\W+', query.lower()) if len(w) >= 3 and w not in stop]

    ranked_idx = list(np.argsort(sims)[::-1])
    picked = []
    seen_keys = set()

    action_tokens = ('step', 'settings', 'configure', 'install', 'download', 'app store', 'google play', 'passcode', 'security')
    generic_tokens = ('supported operating system', 'prerequisite', 'minimum requirements')

    is_comparative = bool(re.search(r'\b(difference|compare|versus|vs|between)\b', query.lower()))
    if is_comparative:
        term_freq = {}
        for term in q_terms:
            term_freq[term] = sum(1 for _, sent in candidates if term in sent.lower())
        aspect_terms = [t for t, f in term_freq.items() if f > 0]
        aspect_terms = sorted(aspect_terms, key=lambda t: term_freq[t])[:4]

        for term in aspect_terms:
            best_idx = None
            best_score = -1.0
            for idx, (_, sent) in enumerate(candidates):
                sent_l = sent.lower()
                if term not in sent_l:
                    continue
                score = float(sims[idx])

                if any(tok in sent_l for tok in action_tokens):
                    score += 0.14
                if any(tok in sent_l for tok in generic_tokens):
                    score -= 0.20
                if f'for {term}' in sent_l or f'{term} device' in sent_l or f'{term} devices' in sent_l:
                    score += 0.10

                if score > best_score:
                    best_score = score
                    best_idx = idx

            if best_idx is not None and best_score >= 0.26:
                src, sent = candidates[best_idx]
                key = re.sub(r'^\d+[\)\.:\-\s]+', '', sent.lower())
                if key not in seen_keys:
                    seen_keys.add(key)
                    picked.append((src, f"{term.upper()}: {sent}", best_score))
                    if len(picked) >= max_points:
                        break

    if len(picked) < max_points:
        for idx in ranked_idx:
            src, sent = candidates[idx]
            key = re.sub(r'^\d+[\)\.:\-\s]+', '', sent.lower())
            if key in seen_keys:
                continue
            if float(sims[idx]) < 0.30:
                continue
            seen_keys.add(key)
            picked.append((src, sent, float(sims[idx])))
            if len(picked) >= max_points:
                break

    if not picked:
        return 'Not found in provided context.'

    lines = [f"- {sent} [Source {src}]" for src, sent, _ in picked]
    return '\n'.join(lines)

def get_cached_text_generation_pipeline(model_name='Qwen/Qwen1.5-0.5B-Chat', cache_prefix='local_gen'):
    cache_key = f'{cache_prefix}::{model_name}'
    pipe = _GLOBAL_MODELS.get(cache_key)
    if pipe is None:
        pipe = pipeline('text-generation', model=model_name)
        _GLOBAL_MODELS[cache_key] = pipe
    return pipe

def generate_answer_hf(prompt, hf_model='Qwen/Qwen2.5-7B-Instruct'):
    from huggingface_hub import InferenceClient

    hf_token = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
    candidate_models = [hf_model, 'meta-llama/Llama-3-8B-Instruct']
    last_error = None

    if hf_token and hf_token != 'your_hf_token_here':
        client = InferenceClient(provider='hf-inference', api_key=hf_token)
        for model_name in candidate_models:
            start = time.time()
            try:
                out = client.text_generation(
                    prompt,
                    model=model_name,
                    max_new_tokens=450,
                    temperature=0.2
                )
                latency = time.time() - start
                return _clean_generated_answer(out), latency
            except Exception as e:
                last_error = repr(e)
                continue

    # extractive_start = time.time()
    # extractive_answer = _extractive_fallback_answer(prompt, max_points=6)
    # if extractive_answer:
    #     return extractive_answer, time.time() - extractive_start

    local_gen_pipe = get_cached_text_generation_pipeline(model_name='Qwen/Qwen1.5-0.5B-Chat', cache_prefix='local_gen')

    local_prompt = prompt
    try:
        tok = local_gen_pipe.tokenizer
        max_positions = int(getattr(local_gen_pipe.model.config, 'n_positions', 1024))
        max_input_tokens = max(64, max_positions - 120)
        token_ids = tok.encode(prompt, add_special_tokens=False)
        if len(token_ids) > max_input_tokens:
            token_ids = token_ids[-max_input_tokens:]
            local_prompt = tok.decode(token_ids, skip_special_tokens=True)
    except Exception:
        pass

    start = time.time()
    local_out = local_gen_pipe(
        local_prompt,
        max_new_tokens=512,
        do_sample=True,
        repetition_penalty=1.15,   # Forces the model to use new words instead of repeating the same sentence structure
        temperature=0.4,           # slightly higher to encourage natural language
        pad_token_id=local_gen_pipe.tokenizer.eos_token_id,
        return_full_text=False
    )
    latency = time.time() - start
    if isinstance(local_out, list) and len(local_out) > 0:
        if 'generated_text' in local_out[0]:
            return _clean_generated_answer(local_out[0]['generated_text']), latency
        return _clean_generated_answer(str(local_out[0])), latency

    if last_error:
        return f'Fallback output unavailable. Remote error: {last_error}', latency
    return _clean_generated_answer(str(local_out)), latency

### Logging

In [15]:
class RAGLogger:
    def __init__(self):
        self.logger = logging.getLogger('rag_system')
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
            self.logger.addHandler(handler)

    def log_rag_interaction(self, query, retrieved_chunks, response, metrics=None, user_feedback=None):
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'query': query,
            'retrieved_chunks': [
                {
                    'id': chunk.get('id', 'na'),
                    'score': chunk.get('score', chunk.get('rerank_score', 0.0)),
                    'source': chunk.get('source', 'unknown')
                } for chunk in retrieved_chunks
            ],
            'response': response,
            'metrics': metrics or {},
            'user_feedback': user_feedback,
            'retrieval_count': len(retrieved_chunks)
        }
        self.logger.info(f'RAG_INTERACTION: {log_entry}')

    def analyze_failures(self):
        # Extend later by querying persisted logs
        pass

### Corpus Loading and Indexing

In [16]:
def upsert_chunks_to_pinecone(retriever: HybridRetriever, chunks, batch_size=100):
    if retriever.pc_index is None:
        if not _RUNTIME_FLAGS.get('pinecone_warning_printed', False):
            print('Pinecone not configured. Set PINECONE_API_KEY and retry.')
            _RUNTIME_FLAGS['pinecone_warning_printed'] = True
        return

    vectors = []
    for chunk in chunks:
        vec = retriever.embedding_model.encode(chunk['text']).tolist()
        vectors.append({
            'id': chunk['id'],
            'values': vec,
            'metadata': {'text': chunk['text'], 'source': chunk['source']}
        })

    for i in range(0, len(vectors), batch_size):
        retriever.pc_index.upsert(vectors=vectors[i:i + batch_size])

def save_chunks_to_cache(chunks, cache_path):
    import json

    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    with cache_path.open('w', encoding='utf-8') as f:
        for chunk in chunks:
            chunk_id = str(chunk.get('id', '')).strip()
            text = str(chunk.get('text', '')).strip()
            source = str(chunk.get('source', 'unknown')).strip() or 'unknown'
            if chunk_id and text:
                record = {'id': chunk_id, 'text': text, 'source': source}
                f.write(json.dumps(record, ensure_ascii=False))
                f.write('\n')

def load_chunks_from_cache(cache_path):
    import json

    cache_path = Path(cache_path)
    if not cache_path.exists():
        return []

    chunks = []
    with cache_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            chunk_id = str(record.get('id', '')).strip()
            text = str(record.get('text', '')).strip()
            source = str(record.get('source', 'unknown')).strip() or 'unknown'
            if chunk_id and text:
                chunks.append({'id': chunk_id, 'text': text, 'source': source})

    return chunks

def prepare_retrieval_indexes(corpus_path, chunking_method='recursive', force_rechunk=False, use_in_memory_cache=True):
    cache_path = get_chunk_cache_path(corpus_path, chunking_method=chunking_method)
    memory_key = str(cache_path)

    if force_rechunk:
        _RETRIEVAL_CACHE.pop(memory_key, None)

    if use_in_memory_cache and not force_rechunk and memory_key in _RETRIEVAL_CACHE:
        cached = _RETRIEVAL_CACHE[memory_key]
        print(f'Using in-memory retriever cache: {cache_path.name}')
        return cached['retriever'], cached['docs'], cached['chunks']

    docs = read_corpus_documents(corpus_path)
    if len(docs) == 0:
        raise ValueError('No documents found. Add files in your corpus path and rerun.')

    chunks = []
    if not force_rechunk:
        chunks = load_chunks_from_cache(cache_path)
        if len(chunks) > 0:
            print(f'Loaded {len(chunks)} chunks from cache: {cache_path}')
            print('Reusing cached chunks. Skipping chunking step.')

    if len(chunks) == 0:
        chunks = build_chunks_from_docs(docs, method=chunking_method)
        save_chunks_to_cache(chunks, cache_path)
        print(f'Chunked corpus and saved {len(chunks)} chunks to cache: {cache_path}')

    retriever = HybridRetriever()
    retriever.set_bm25_corpus(chunks)
    upsert_chunks_to_pinecone(retriever, chunks)

    _RETRIEVAL_CACHE[memory_key] = {'retriever': retriever, 'docs': docs, 'chunks': chunks}

    print(f'Documents loaded: {len(docs)}')
    print(f'Chunks ready: {len(chunks)}')
    return retriever, docs, chunks

### LLM-as-a-Judge Evaluation (Faithfulness + Relevancy)

In [17]:
def call_hf_judge(prompt, model='meta-llama/Meta-Llama-3-8B-Instruct'):
    from huggingface_hub import InferenceClient

    hf_token = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
    candidate_models = [model, 'Qwen/Qwen2.5-7B-Instruct']
    last_error = None

    use_remote = bool(hf_token) and hf_token != 'your_hf_token_here' and not _RUNTIME_FLAGS.get('disable_hf_judge_remote', False)
    if use_remote:
        client = InferenceClient(provider='hf-inference', api_key=hf_token)
        for model_name in candidate_models:
            try:
                out = client.text_generation(
                    prompt,
                    model=model_name,
                    max_new_tokens=128,
                    temperature=0.0
                )
                return out
            except Exception as e:
                last_error = repr(e)
                continue
        _RUNTIME_FLAGS['disable_hf_judge_remote'] = True

    local_judge_pipe = get_cached_text_generation_pipeline(model_name='Qwen/Qwen1.5-0.5B-Chat', cache_prefix='local_judge')
    local_out = local_judge_pipe(
        prompt[:600],
        max_new_tokens=48,
        do_sample=False,
        pad_token_id=local_judge_pipe.tokenizer.eos_token_id,
        return_full_text=False
    )
    if isinstance(local_out, list) and len(local_out) > 0 and 'generated_text' in local_out[0]:
        return local_out[0]['generated_text']

    if last_error:
        return f'Fallback output unavailable. Remote error: {last_error}'
    return str(local_out)

def _tokenize_for_coverage(text: str) -> set:
    return {w for w in re.split(r'\W+', (text or '').lower()) if len(w) > 2}

def faithfulness_score(answer_text, retrieved_chunks, embedding_model=None, support_threshold=0.54):
    """Embedding-based faithfulness: each answer sentence must be semantically supported by retrieved context."""
    if embedding_model is None:
        embedding_model = get_cached_embedding_model('all-MiniLM-L6-v2')

    answer_sentences = [s for s in _split_sentences(answer_text) if len(s.split()) >= 4]
    if not answer_sentences:
        single = normalize_text(answer_text)
        answer_sentences = [single] if single else []

    context_sentences = []
    for chunk in retrieved_chunks:
        ctext = normalize_text(chunk.get('text', ''))
        if not ctext:
            continue
        split = [s for s in _split_sentences(ctext) if len(s.split()) >= 4]
        context_sentences.extend(split if split else [ctext])

    if not answer_sentences or not context_sentences:
        return 0.0, []

    context_emb = np.array(embedding_model.encode(context_sentences, show_progress_bar=False))
    answer_emb = np.array(embedding_model.encode(answer_sentences, show_progress_bar=False))

    verdicts = []
    for sent, emb in zip(answer_sentences, answer_emb):
        sims = cosine_similarity([emb], context_emb)[0]
        support_score = float(np.max(sims))
        verdicts.append({
            'claim': sent,
            'support_score': support_score,
            'supported': bool(support_score >= support_threshold)
        })

    faith = float(np.mean([1.0 if v['supported'] else 0.0 for v in verdicts]))
    return faith, verdicts

def relevancy_score(original_query, answer_text, embedding_model=None, retrieved_chunks=None):
    """Stable relevancy metric that does not rely on judge generation quality."""
    if embedding_model is None:
        embedding_model = get_cached_embedding_model('all-MiniLM-L6-v2')

    q_text = normalize_text(original_query)
    a_text = normalize_text(answer_text)
    if not q_text or not a_text:
        return 0.0, [0.0, 0.0, 0.0]

    q_vec = embedding_model.encode(q_text)
    a_vec = embedding_model.encode(a_text)
    qa_similarity = float(cosine_similarity([q_vec], [a_vec])[0][0])
    qa_similarity = float(np.clip(qa_similarity, 0.0, 1.0))

    context_similarity = 0.0
    if retrieved_chunks:
        context_texts = [normalize_text(c.get('text', '')) for c in retrieved_chunks]
        context_texts = [t for t in context_texts if t]
        if context_texts:
            ctx_emb = np.array(embedding_model.encode(context_texts, show_progress_bar=False))
            context_similarity = float(np.max(cosine_similarity([a_vec], ctx_emb)[0]))
            context_similarity = float(np.clip(context_similarity, 0.0, 1.0))

    q_terms = _tokenize_for_coverage(q_text)
    a_terms = _tokenize_for_coverage(a_text)
    lexical_coverage = float(len(q_terms & a_terms) / len(q_terms)) if q_terms else 0.0

    relevancy = float(np.clip(
        0.6 * qa_similarity + 0.3 * context_similarity + 0.1 * lexical_coverage,
        0.0,
        1.0
    ))
    return relevancy, [qa_similarity, context_similarity, lexical_coverage]

In [18]:
def _expand_queries_for_retrieval(query: str) -> list:
    query = normalize_text(query)
    queries = [query]
    ql = query.lower()

    if re.search(r'\b(difference|compare|versus|vs|between)\b', ql):
        known_aspects = ['ios', 'android', 'windows', 'iphone', 'ipad']
        aspects = [a for a in known_aspects if a in ql]
        for aspect in aspects:
            queries.append(f'configure {aspect} device company email steps')
            queries.append(f'{aspect} company email setup process')

    dedup = []
    seen = set()
    for q in queries:
        key = q.lower()
        if key not in seen:
            seen.add(key)
            dedup.append(q)
    return dedup

def _extract_aspect_terms(query: str) -> list:
    ql = query.lower()
    known_aspects = ['ios', 'android', 'windows', 'iphone', 'ipad', 'vpn', 'exchange']
    return [a for a in known_aspects if a in ql]

def run_single_query_pipeline(query, retriever, retrieval_mode='hybrid', top_k=5):
    t0 = time.time()
    retrieval_pool = max(top_k * 4, 24)
    expanded_queries = _expand_queries_for_retrieval(query)

    merged = {}
    for q in expanded_queries:
        if retrieval_mode == 'semantic':
            local = retriever.retrieve_semantic_only(q, top_k=retrieval_pool)
        else:
            local = retriever.retrieve_hybrid(q, top_k=retrieval_pool, rerank=True)

        for item in local:
            cid = item.get('id')
            if not cid:
                continue
            score = float(item.get('rerank_score', item.get('score', 0.0)))
            prev = merged.get(cid)
            if prev is None or score > float(prev.get('rerank_score', prev.get('score', 0.0))):
                merged[cid] = dict(item)

    retrieved = list(merged.values())
    retrieved.sort(key=lambda x: float(x.get('rerank_score', x.get('score', 0.0))), reverse=True)
    retrieved = remove_duplicates(retrieved)
    retrieved = diversify_by_source(retrieved, max_per_source=4)
    retrieval_latency = time.time() - t0

    aspect_terms = _extract_aspect_terms(query)
    forced_aspect_chunks = []
    forced_ids = set()
    action_tokens = ('step', 'settings', 'configure', 'install', 'download', 'app store', 'google play', 'passcode', 'security')
    generic_tokens = ('supported operating system', 'prerequisite', 'minimum requirements')

    for term in aspect_terms:
        best_chunk = None
        best_score = -1.0
        for c in retrieved:
            text_l = str(c.get('text', '')).lower()
            cid = c.get('id')
            if term not in text_l or cid in forced_ids:
                continue

            score = float(c.get('rerank_score', c.get('score', 0.0)))
            if any(tok in text_l for tok in action_tokens):
                score += 0.10
            if any(tok in text_l for tok in generic_tokens):
                score -= 0.16
            if f'for {term}' in text_l or f'{term} device' in text_l or f'{term} devices' in text_l:
                score += 0.08

            if score > best_score:
                best_score = score
                best_chunk = c

        if best_chunk is not None:
            forced_aspect_chunks.append(best_chunk)
            forced_ids.add(best_chunk.get('id'))

    auto_selected = intelligent_context_selection(retrieved, retriever.embedding_model, max_tokens=3200)
    selected_chunks = []
    selected_ids = set()
    for c in forced_aspect_chunks + auto_selected:
        cid = c.get('id')
        if cid not in selected_ids:
            selected_chunks.append(c)
            selected_ids.add(cid)

    if len(selected_chunks) < 6:
        for c in retrieved:
            cid = c.get('id')
            if cid not in selected_ids:
                selected_chunks.append(c)
                selected_ids.add(cid)
            if len(selected_chunks) >= 6:
                break

    if not selected_chunks:
        selected_chunks = retrieved[:min(6, len(retrieved))]

    prompt = create_rag_prompt(query, selected_chunks)
    answer, generation_latency = generate_answer_hf(prompt)

    faith_score, claim_verification = faithfulness_score(
        answer,
        selected_chunks,
        embedding_model=retriever.embedding_model
    )
    rel_score, rel_sims = relevancy_score(
        query,
        answer,
        embedding_model=retriever.embedding_model,
        retrieved_chunks=selected_chunks
    )

    if (faith_score < 0.55 or rel_score < 0.55) and selected_chunks:
        strict_prompt = prompt + "\n\nIMPORTANT: Cover all major aspects in the question with explicit source tags."
        retry_answer, retry_latency = generate_answer_hf(strict_prompt)
        retry_faith, retry_claims = faithfulness_score(
            retry_answer,
            selected_chunks,
            embedding_model=retriever.embedding_model
        )
        retry_rel, retry_rel_sims = relevancy_score(
            query,
            retry_answer,
            embedding_model=retriever.embedding_model,
            retrieved_chunks=selected_chunks
        )

        if (retry_faith + retry_rel) > (faith_score + rel_score):
            answer = retry_answer
            generation_latency += retry_latency
            faith_score = retry_faith
            rel_score = retry_rel
            claim_verification = retry_claims
            rel_sims = retry_rel_sims

    total_latency = retrieval_latency + generation_latency
    metrics = {
        'faithfulness': faith_score,
        'relevancy': rel_score,
        'retrieval_latency': retrieval_latency,
        'generation_latency': generation_latency,
        'total_latency': total_latency
    }

    return {
        'query': query,
        'answer': answer,
        'retrieved_chunks': selected_chunks,
        'claim_verification': claim_verification,
        'relevancy_similarities': rel_sims,
        'metrics': metrics
    }

### Ablation Study (Chunking + Retrieval)

In [19]:
def run_ablation(test_queries, corpus_path):
    settings = [
        # {'chunking': 'arbitary', 'retrieval': 'semantic'},
        # {'chunking': 'recursive', 'retrieval': 'semantic'},
        # {'chunking': 'recursive', 'retrieval': 'hybrid'},
        {'chunking': 'semantic', 'retrieval': 'hybrid'}
    ]

    rows = []
    for cfg in settings:
        retriever, _, _ = prepare_retrieval_indexes(corpus_path, chunking_method=cfg['chunking'])

        faithfulness_vals = []
        relevancy_vals = []
        latency_vals = []

        for q in test_queries:
            out = run_single_query_pipeline(
                query=q,
                retriever=retriever,
                retrieval_mode=cfg['retrieval'],
                top_k=TOP_K
            )
            faithfulness_vals.append(out['metrics']['faithfulness'])
            relevancy_vals.append(out['metrics']['relevancy'])
            latency_vals.append(out['metrics']['total_latency'])

        rows.append({
            'chunking': cfg['chunking'],
            'retrieval': cfg['retrieval'],
            'faithfulness_avg': float(np.mean(faithfulness_vals)),
            'relevancy_avg': float(np.mean(relevancy_vals)),
            'latency_avg_sec': float(np.mean(latency_vals))
        })

    return pd.DataFrame(rows).sort_values(['faithfulness_avg', 'relevancy_avg'], ascending=False)

In [20]:
# Example fixed test queries for assignment evaluation (10-20 recommended)
TEST_QUERIES = [
    # --- Caregiver Burnout & Family Stress (Tests context retrieval for specific life events) ---
    "My mother was recently diagnosed with Alzheimer's and I am her primary caregiver. I am completely exhausted trying to balance her needs with my job and my own family. How do I cope with this burnout?",
    "I've been dealing with my parents' tumultuous relationship and constant bickering for five years. It's taking a huge toll on my mental and physical well-being. How can I set boundaries?",

    # --- Treatment Resistance & Hopelessness (Tests handling of negative/frustrated sentiment) ---
    "I've been struggling with my mental health for a while. I've tried visualization, positive thinking, and even medication, but nothing seems to work. I feel like I'm drowning in confusion. What should I do next?",

    # --- Isolation & Social Anxiety (Tests semantic matching for emotional states) ---
    "I often feel unhappy doing things alone and the isolation is suffocating. I want to reach out to friends, but I always feel like I am a burden to them. How can I overcome this feeling of emptiness?",
    "I'm starting to feel like I'll never be able to form close relationships again. I joined new social groups but still can't make meaningful connections. What am I doing wrong?",

    # --- Specific Conditions & Panic (Tests BM25/Keyword retrieval for specific clinical terms) ---
    "My spouse has bipolar disorder and we recently had to change their treatment plan. I live in constant fear that they will relapse and we'll be back to square one. How do I manage my own anxiety about their condition?",
    "I am constantly overwhelmed by intrusive thoughts and fear of mortality. It triggers severe panic attacks that disrupt my sleep. What grounding techniques or muscle relaxation exercises can help me?"
]

# Run these after setting CORPUS_PATH and API keys
retriever, docs, chunks = prepare_retrieval_indexes(CORPUS_PATH, chunking_method=CHUNKING_METHOD)
sample_output = run_single_query_pipeline(TEST_QUERIES[0], retriever, retrieval_mode='hybrid', top_k=TOP_K)

print('\n=== SAMPLE QUERY ===')
print(sample_output['query'])
print('\n=== GENERATED ANSWER ===')
print(sample_output['answer'])
print('\n=== METRICS ===')
print(sample_output['metrics'])
print('\n=== TOP CONTEXT CHUNKS (preview) ===')
for i, c in enumerate(sample_output['retrieved_chunks'][:3], start=1):
    print(f'[{i}]', c.get('text', '')[:250], '...')

ablation_df = run_ablation(TEST_QUERIES, CORPUS_PATH)
ablation_df

Chunked corpus and saved 3045 chunks to cache: /content/chunk_cache/support_1000_recursive_v6_1973ade220d5.jsonl
Pinecone not configured. Set PINECONE_API_KEY and retry.
Documents loaded: 1000
Chunks ready: 3045

=== SAMPLE QUERY ===
My mother was recently diagnosed with Alzheimer's and I am her primary caregiver. I am completely exhausted trying to balance her needs with my job and my own family. How do I cope with this burnout?

=== GENERATED ANSWER ===
ANSWER: Warmly, professionally mental health counselor speaking directly to the patient: Hello! Thank you for sharing your story about your mother's recent diagnosis with Alzheimer's. As a primary caregiver, I understand the challenges you face balancing her needs with your own job and family. Here are three practical coping strategies that may help you manage your burnout: 1. Prioritize Self-Care: Make sure to schedule regular breaks throughout the day to rest and recharge. Set aside time for activities that you enjoy, such as readin

,chunking,retrieval,faithfulness_avg,relevancy_avg,latency_avg_sec
0,semantic,hybrid,0.871088,0.575812,11.297213


In [21]:
# Temporary fast-mode run (runtime only). Restores originals at the end.
_orig_generate_answer_hf = generate_answer_hf
_orig_intelligent_context_selection = intelligent_context_selection

def _generate_answer_hf_fast(prompt, hf_model='Qwen/Qwen2.5-7B-Instruct'):
    from huggingface_hub import InferenceClient

    hf_token = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
    candidate_models = [hf_model, 'meta-llama/Llama-3-8B-Instruct']
    last_error = None

    if hf_token and hf_token != 'your_hf_token_here':
        client = InferenceClient(provider='hf-inference', api_key=hf_token)
        for model_name in candidate_models:
            start = time.time()
            try:
                out = client.text_generation(
                    prompt,
                    model=model_name,
                    max_new_tokens=220,
                    temperature=0.2
                )
                latency = time.time() - start
                return _clean_generated_answer(out), latency
            except Exception as e:
                last_error = repr(e)
                continue

    local_gen_pipe = get_cached_text_generation_pipeline(model_name='Qwen/Qwen1.5-0.5B-Chat', cache_prefix='local_gen')

    local_prompt = prompt
    try:
        tok = local_gen_pipe.tokenizer
        max_positions = int(getattr(local_gen_pipe.model.config, 'n_positions', 1024))
        max_input_tokens = max(64, max_positions - 120)
        token_ids = tok.encode(prompt, add_special_tokens=False)
        if len(token_ids) > max_input_tokens:
            token_ids = token_ids[-max_input_tokens:]
            local_prompt = tok.decode(token_ids, skip_special_tokens=True)
    except Exception:
        pass

    start = time.time()
    local_out = local_gen_pipe(
        local_prompt,
        max_new_tokens=160,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=local_gen_pipe.tokenizer.eos_token_id,
        return_full_text=False
    )
    latency = time.time() - start

    if isinstance(local_out, list) and len(local_out) > 0:
        if 'generated_text' in local_out[0]:
            return _clean_generated_answer(local_out[0]['generated_text']), latency
        return _clean_generated_answer(str(local_out[0])), latency

    if last_error:
        return f'Fallback output unavailable. Remote error: {last_error}', latency
    return _clean_generated_answer(str(local_out)), latency

def _intelligent_context_selection_fast(retrieved_chunks, embedding_model, max_tokens=1500):
    return _orig_intelligent_context_selection(
        retrieved_chunks,
        embedding_model,
        max_tokens=min(max_tokens, 1000)
    )

generate_answer_hf = _generate_answer_hf_fast
intelligent_context_selection = _intelligent_context_selection_fast

try:
    retriever, docs, chunks = prepare_retrieval_indexes(CORPUS_PATH, chunking_method=CHUNKING_METHOD)
    target_query = TEST_QUERIES[0]

    sample_output = run_single_query_pipeline(target_query, retriever, retrieval_mode='hybrid', top_k=TOP_K)
    print('\n=== FAST SAMPLE QUERY ===')
    print(sample_output['query'])
    print('\n=== FAST SAMPLE METRICS ===')
    print(sample_output['metrics'])

    ablation_df_one = run_ablation([target_query], CORPUS_PATH)
    print('\n=== ONE-QUERY ABLATION ===')
    display(ablation_df_one)
finally:
    generate_answer_hf = _orig_generate_answer_hf
    intelligent_context_selection = _orig_intelligent_context_selection
    print('\nRestored original generation/context-selection functions.')

Using in-memory retriever cache: support_1000_recursive_v6_1973ade220d5.jsonl

=== FAST SAMPLE QUERY ===
My mother was recently diagnosed with Alzheimer's and I am her primary caregiver. I am completely exhausted trying to balance her needs with my job and my own family. How do I cope with this burnout?

=== FAST SAMPLE METRICS ===
{'faithfulness': 0.0, 'relevancy': 0.02117880955338478, 'retrieval_latency': 0.2447049617767334, 'generation_latency': 6.053479909896851, 'total_latency': 6.298184871673584}
Using in-memory retriever cache: support_1000_semantic_v6_4c5a73e35557.jsonl

=== ONE-QUERY ABLATION ===


,chunking,retrieval,faithfulness_avg,relevancy_avg,latency_avg_sec
0,semantic,hybrid,0.0,0.021179,6.571246



Restored original generation/context-selection functions.
